# Embed and Index the DS Corpus

Takes a completed corpus build from `1_download_ds_papers.ipynb` (a dated
`output/ds_include_pipeline_<date>/` directory), selects the papers to index
(`n_sections >= MIN_SECTIONS`, default 4, AND `type == "article"`), obtains their
embedding vectors, and inserts them into the Milvus collection the retrieval service
uses.

Embedding vectors are reuse-first: if a vector pickle for a PMID already exists in any
earlier embed run and that run embedded the identical JSON text (verified by inode or
content hash), it is reused with no API call; only papers without a valid prior pickle
are computed fresh through the DeepInfra API. A rerun over unchanged papers therefore
completes in minutes and costs nothing.

## How the document count comes about

The collection stores ONLY papers that satisfy every criterion, so the stored count,
the searchable count, and the /health count are the same number. For the 2026-09-02
snapshot:

    2,075  records in the PMC query snapshot (input/PMC_DownSynd_09_02_2026.txt)
    1,891  eligible after the metadata gate (excluded publication types out,
           PMCID present, not retracted per MEDLINE tag + Crossref)
    1,837  retained after the full-text criterion (PMC OAI serves a usable body)
    1,669  with >= MIN_SECTIONS (4) of the six target sections and at least one
           retrieval section (this notebook, Cell 2, criterion 1)
    1,594  pub_type == "article": original research only; drops 64 reviews,
           8 preprints, 2 errata, 1 other (this notebook, Cell 2, criterion 2)

Steps 1 to 3 are notebook 1's screening funnel (per-paper audit in
`step2_screening_audit.csv`); steps 4 and 5 are this notebook's selection from
`master_table.csv`. Result: 1,594 papers, 65,677 chunks, 158 INCLUDE-funded.

## How to compute the embedding vectors again

1. In Cell 1, set `RUN_DATE` to the corpus build to embed and `MIN_SECTIONS` to the
   selection threshold. `SNAPSHOT_DATE` (the MEDLINE export date the corpus derives
   from) determines the collection name.
2. Ensure `DEEPINFRA_API_KEY` is set in `srv/.env` (no GPU and no local model needed).
3. Run Cells 1 to 6 top to bottom in the `dschat` kernel from `srv/nbs`.
4. Cell 3 reuses before it recalculates: unchanged papers are hardlinked in; genuinely
   new or changed papers take their share of the roughly 25 to 30 minutes a full
   1,800-paper embed costs. Resume-safe: an interrupted run continues where it stopped.
   To force a fresh recompute, delete the relevant pickles or pass `force=True`.

Note on vector identity: embeddings come from a hosted API, so re-embedding the same
text is not guaranteed bit-identical run to run. Reuse of existing pickles is exact,
which is another reason it is preferred whenever the text is unchanged.

## Design decisions, each deliberate

- Selection is by section count AND publication type from the build's master table;
  papers the retrieval filter could never reach (too few sections, or a non-article
  pub type) are excluded before any embedding or storage cost, so stored == searchable
  and the API's `pub_type == "article"` filter is a no-op safety net.
- The production pipeline is reused, not reimplemented: `embed_and_insert.py` does the
  chunking (RecursiveCharacterTextSplitter, 1,200 characters, 200 overlap), the
  DeepInfra embedding calls (nvidia/llama-nemotron-embed-vl-1b-v2, 2,048-dim), the
  deterministic primary keys, and the 31-field schema.
- The collection is named for the input MEDLINE snapshot date
  (`DS_INCLUDE_<snapshot>_DEEPINFRA_2048`), because the snapshot, not the rerun date,
  defines the corpus.
- Insertion is idempotent per chunk (deterministic keys, delete-before-insert), so
  rerunning the notebook converges to the same state instead of duplicating. The target
  is the collection the API serves; a rerun with an unchanged corpus is therefore safe,
  and a corpus change should be followed by an API cache clear. Note the upsert never
  DELETES papers that drop out of the selection; removing papers already in the
  collection takes an explicit purge (see `scripts/purge_non_articles_2026_09_06.py`
  for the 2026-09-06 non-article purge) or a rebuild into a fresh collection.
- The superseded pre-selection corpus is preserved under an `_ARCHIVED` collection name
  (`scripts/rename_collections_2026_09_06.py` with `ROLLBACK = True` restores it).

Environment: the `dschat` kernel. Run from `srv/nbs`.

Current corpus: 1,594 original research articles with 4 or more sections from the
2026-09-02 snapshot, 65,677 vectors, 158 INCLUDE-funded. All selected papers passed
the dual retraction check (MEDLINE tag plus Crossref); none retracted.


## Manual run quickstart

From `srv/nbs` in the `dschat` kernel, run all cells top to bottom (Cells 1 to 6).
Milvus must be running and `DEEPINFRA_API_KEY` set in `srv/.env` (only the Cell 6 query
embedding actually uses it). Total time roughly 3 to 6 minutes for an unchanged corpus.

Note: the target `DS_INCLUDE_2026_09_02_DEEPINFRA_2048` is the live production
collection (the name carries the input snapshot date of `PMC_DownSynd_09_02_2026.txt`).
Rerunning with an unchanged corpus is safe: the idempotent insert converges to the
identical state. After any run that changes content, clear the API cache:
`curl -X POST http://localhost:8698/cache/clear`.

Expected output per cell for the current corpus (`MIN_SECTIONS = 4`, articles only):

- Cell 1: build `ds_include_pipeline_2026-09-02`, collection
  `DS_INCLUDE_2026_09_02_DEEPINFRA_2048`.
- Cell 2: 1,594 of 1,837 papers selected (168 excluded for fewer sections, 75 for a
  non-article pub type), 158 INCLUDE-tagged among them.
- Cell 3: all 1,594 pickles already present in the pickle directory, nothing left to
  compute. Seconds, zero API cost.
- Cell 4: 1,594 pickles, 65,677 chunks, zero problems (unpickles everything, allow a
  couple of minutes).
- Cell 5: insert, about a minute, converging to the existing state.
- Cell 6: 65,677 vectors, 1,594 documents, 158 INCLUDE docs, and a live search
  returning 5 hits with top score above 0.4. A slightly short count right after the
  insert is growing-segment settling; wait a minute and rerun the cell.

Because only papers satisfying every criterion are stored (including
`pub_type == "article"`), Cell 6, `/health`, and the GUI sidebar/About/FAQ all report
the same numbers: 1,594 documents, 158 INCLUDE, 65,677 vectors. The full derivation of
1,594 is at the top of this notebook ("How the document count comes about").

For a future corpus (a new MEDLINE snapshot), set `SNAPSHOT_DATE` and `RUN_DATE` in
Cell 1; the new snapshot date yields a new collection name, and cutover to it is the
deliberate manual step described at the end of this notebook.


In [ ]:
# ---- Cell 1: configuration ----
import json, os, sys, time
from pathlib import Path

import pandas as pd

NB_DIR = Path.cwd()
SRV = NB_DIR.parent if NB_DIR.name == "nbs" else NB_DIR
assert (SRV / "scripts" / "embed_and_insert.py").exists(), f"run from srv/nbs (SRV={SRV})"

# Which corpus build to embed: a dated output directory from 1_download_ds_papers.
# The 2026-09-02 directory is the canonical build for the 2026-09-02 snapshot (the
# later 2026-09-05 rerun produced identical content, was hardlink-merged into it,
# and has been archived).
RUN_DATE = "2026-09-02"

# Paper selection, applied in Cell 2. Two criteria:
#   1. n_sections >= MIN_SECTIONS of the six target sections (Abstract, Introduction,
#      Methods, Results, Discussion, Conclusion). 4 or more guarantees at least two
#      retrieval sections.
#   2. pub_type == "article": original research only. Reviews, preprints and errata
#      are excluded so the collection stores exactly what retrieval can search
#      (the API's pub_type filter becomes a no-op safety net).
# 2026-09-02 build: 1,837 papers -> 1,669 with >= 4 sections -> 1,594 articles.
MIN_SECTIONS = 4

RUN_DIR    = SRV / "output" / f"ds_include_pipeline_{RUN_DATE}"
STAGE_DIR  = SRV / "output" / f"_embed_input_ge{MIN_SECTIONS}sections_articles_{RUN_DATE}"
PICKLE_DIR = (SRV / "output" / "embeddings_pickle"
              / f"ds_include_{RUN_DATE}_ge{MIN_SECTIONS}sections_articles_deepinfra_2048")

# The collection is named for the INPUT SNAPSHOT date, i.e. the MEDLINE export the
# corpus derives from (input/PMC_DownSynd_09_02_2026.txt), not the pipeline rerun date.
# This is the production collection served by the API (.env MILVUS_COLLECTION):
# insertion is idempotent (deterministic keys, delete-before-insert), so rerunning this
# notebook with an unchanged corpus converges to the identical state. The superseded
# pre-selection corpus is preserved under the _ARCHIVED name; never target it.
SNAPSHOT_DATE = "2026-09-02"
NEW_COLLECTION = f"DS_INCLUDE_{SNAPSHOT_DATE.replace('-', '_')}_DEEPINFRA_2048"

assert RUN_DIR.exists(), f"corpus build not found: {RUN_DIR}"
assert (RUN_DIR / "master_table.csv").exists(), "run Part C of the pipeline notebook first"
print("corpus build :", RUN_DIR.name)
print("pickles      :", PICKLE_DIR)
print("collection   :", NEW_COLLECTION)


## Cell 2: select the papers to index

Read from the build's `master_table.csv`, which Part C of the download notebook wrote
after all criteria and the INCLUDE tagging. Two selection criteria:

1. `n_sections >= MIN_SECTIONS` (default 4, which also guarantees at least two of the
   four retrieval sections). Papers below the threshold, including abstract-only
   records the retrieval filter could never reach, are excluded up front.
2. `type == "article"`: original research only. Reviews, preprints and errata that
   survived the download gate are not indexed, so the collection stores exactly the
   set retrieval can search and /health reports the same number the collection holds.

The staged directory is symlinks, not copies, so the JSON cache remains the single
source of truth and INCLUDE tags cannot drift between the two.


In [13]:
# ---- Cell 2: stage the selected papers ----
import shutil

m = pd.read_csv(RUN_DIR / "master_table.csv")
enough_sections = (m["n_sections"] >= MIN_SECTIONS) & (m["n_retrieval"] > 0)
is_article = m["type"] == "article"
keep = m[enough_sections & is_article].copy()
print(f"papers in build                    : {len(m)}")
print(f"excluded (fewer sections)          : {int((~enough_sections).sum())}")
print(f"excluded (non-article pub type)    : {int((enough_sections & ~is_article).sum())} "
      f"{dict(m.loc[enough_sections & ~is_article, 'type'].value_counts())}")
print(f"selected (article, >= {MIN_SECTIONS} sections)  : {len(keep)}")
print(f"INCLUDE-tagged among selected      : {int(keep['is_include'].sum())}")

if STAGE_DIR.exists():
    shutil.rmtree(STAGE_DIR)
STAGE_DIR.mkdir(parents=True)
n = 0
for pmid in keep["pmid"].astype(str):
    src = RUN_DIR / "json_cache" / f"publication_metadata_PMID_{pmid}.json"
    if src.exists():
        (STAGE_DIR / src.name).symlink_to(src.resolve())
        n += 1
print(f"staged {n} JSONs -> {STAGE_DIR.name}")
assert n == len(keep), "some selected papers are missing from the json_cache"


papers in build                    : 1837
excluded (fewer sections)          : 168
excluded (non-article pub type)    : 75 {'review': 64, 'preprint': 8, 'erratum': 2, 'other': 1}
selected (article, >= 4 sections)  : 1594
INCLUDE-tagged among selected      : 158
staged 1594 JSONs -> _embed_input_ge4sections_articles_2026-09-05


## Cell 3: compute the embedding vectors

Reuse first, recalculate only what is missing:

1. Per-PMID reuse pass: if a pickle for the paper already exists in ANY earlier embed
   run under `output/embeddings_pickle/`, and that run embedded the identical JSON
   (verified by inode or content hash against the source build's json_cache), the
   pickle is hardlinked in. No API call, exact same vectors.
2. Only papers with no valid prior pickle go to `compute_embeddings`, which calls the
   DeepInfra API and writes one pickle per paper into `PICKLE_DIR`.

A paper whose JSON text changed since the earlier run fails the identity check and is
recomputed, so reuse can never serve stale vectors. Resume-safe as before: rerunning
this cell fills gaps only. To force recomputation of specific papers, delete their
pickles from `PICKLE_DIR` (and pass `force=True` to also ignore same-directory
pickles). For a deliberate full-corpus recompute, use a fresh `PICKLE_DIR` and skip
the reuse pass by clearing `PRIOR_PICKLE_DIRS` after it is defined.

Cost through DeepInfra is negligible; no GPU or local model is involved.


In [14]:
# ---- Cell 3: compute ----
sys.path.insert(0, str(SRV / "scripts"))
import embed_and_insert as eai

# Per-PMID reuse pass: if an embedding pickle for a paper already exists in ANY earlier
# embed run, and that run embedded the IDENTICAL JSON (same file by inode, or same
# content hash), the pickle is hardlinked in instead of recomputed. Only papers with no
# valid prior pickle are sent to the DeepInfra API below.
import hashlib, os, re as _re

def _json_for(pickle_dir, name):
    m = _re.match(r"ds_include_(\d{4}-\d{2}-\d{2})", pickle_dir.name)
    if not m:
        return None
    return SRV / "output" / f"ds_include_pipeline_{m.group(1)}" / "json_cache" / name

def _same_file(a, b):
    try:
        if a.stat().st_ino == b.stat().st_ino:
            return True
        if a.stat().st_size != b.stat().st_size:
            return False
        return (hashlib.md5(a.read_bytes()).digest()
                == hashlib.md5(b.read_bytes()).digest())
    except OSError:
        return False

PRIOR_PICKLE_DIRS = sorted(
    (d for d in (SRV / "output" / "embeddings_pickle").glob("ds_include_*_deepinfra_2048")
     if d.is_dir() and d != PICKLE_DIR), reverse=True)
PICKLE_DIR.mkdir(parents=True, exist_ok=True)
_reused = 0
for pmid in keep["pmid"].astype(str):
    jname = f"publication_metadata_PMID_{pmid}.json"
    pname = f"publication_metadata_PMID_{pmid}.pkl"
    dest = PICKLE_DIR / pname
    if dest.exists():
        continue
    cur_json = RUN_DIR / "json_cache" / jname
    for prior in PRIOR_PICKLE_DIRS:
        src_pkl = prior / pname
        if not src_pkl.exists() or src_pkl.stat().st_size == 0:
            continue
        prior_json = _json_for(prior, jname)
        if prior_json is None or not prior_json.exists():
            continue
        if _same_file(cur_json, prior_json):
            try:
                os.link(src_pkl, dest)
            except OSError:
                dest.write_bytes(src_pkl.read_bytes())
            _reused += 1
            break
if _reused:
    print(f"embedding pickles reused from earlier runs (identical JSON, no API call): {_reused}")

t0 = time.time()
stats = eai.compute_embeddings(STAGE_DIR, PICKLE_DIR, force=False)
print(f"\ndone in {(time.time()-t0)/60:.1f} min")
n_pkl = len(list(PICKLE_DIR.glob("*.pkl")))
print(f"pickles on disk: {n_pkl}")
assert stats.get("skipped_error", 0) == 0, (
    f"{stats['skipped_error']} papers failed to embed; re-run this cell to retry them "
    "(their pickles are absent, so only the failures are re-attempted)")



----------------------------------------------------------------------
STEP 1: COMPUTE EMBEDDINGS
----------------------------------------------------------------------
Total JSON files    : 1594
Eligible papers     : 1594 (have ≥1 target section)
Pickle directory    : /path/to/project/T21_research_assistant_v7/srv/output/embeddings_pickle/ds_include_2026-09-05_ge4sections_articles_deepinfra_2048

[OK] DeepInfra embeddings initialized (model: nvidia/llama-nemotron-embed-vl-1b-v2, batch: 32)


Computing embeddings: 100%|██████████| 1594/1594 [00:00<00:00, 184032.61it/s]


----------------------------------------------------------------------
EMBEDDING COMPUTATION SUMMARY
----------------------------------------------------------------------
processed           : 0
skipped_empty       : 0
skipped_error       : 0
already_done        : 1,594
total_chunks        : 0
Pickle directory  : /path/to/project/T21_research_assistant_v7/srv/output/embeddings_pickle/ds_include_2026-09-05_ge4sections_articles_deepinfra_2048
----------------------------------------------------------------------


done in 0.0 min
pickles on disk: 1594


## Cell 4: verify the pickles before touching Milvus

Cheap invariants that catch a bad batch before anything is inserted: every pickle
non-empty, every vector 2,048-dim, chunk counts sane, and the INCLUDE tag present on
the expected papers.


In [15]:
# ---- Cell 4: pickle verification ----
import pickle
import numpy as np

pkls = sorted(PICKLE_DIR.glob("*.pkl"))
n_chunks = 0
n_inc_papers = 0
bad = []
for p in pkls:
    rows = pickle.loads(p.read_bytes())
    if not rows:
        bad.append((p.name, "empty")); continue
    if any(len(r["vector"]) != 2048 for r in rows):
        bad.append((p.name, "wrong vector dim")); continue
    n_chunks += len(rows)
    if rows[0].get("is_include"):
        n_inc_papers += 1
print(f"pickles          : {len(pkls)}")
print(f"total chunks     : {n_chunks:,}")
print(f"INCLUDE papers   : {n_inc_papers}")
print(f"problems         : {len(bad)} {bad[:3]}")
assert not bad, "fix the problem pickles before inserting"
assert len(pkls) == len(keep), f"expected {len(keep)} pickles, found {len(pkls)}"


pickles          : 1594
total chunks     : 65,677
INCLUDE papers   : 158
problems         : 0 []


## Cell 5: insert into the NEW collection

The collection name is overridden for this call only; the .env value and the live
service are untouched. `ensure_collection` creates the 31-field schema and IVF_FLAT
index on first use. Insertion is idempotent per chunk (deterministic primary keys with
delete-before-insert), so re-running this cell is safe.


In [16]:
# ---- Cell 5: insert ----
eai.COLLECTION_NAME = NEW_COLLECTION          # target the new collection, not .env's
os.environ["MILVUS_COLLECTION"] = NEW_COLLECTION

t0 = time.time()
ins = eai.insert_embeddings(PICKLE_DIR, mark_include=False)
print(f"insert finished in {(time.time()-t0)/60:.1f} min: {ins}")



----------------------------------------------------------------------
STEP 2: INSERT TO MILVUS
----------------------------------------------------------------------
Pickle files found  : 1594
Pickle directory    : /path/to/project/T21_research_assistant_v7/srv/output/embeddings_pickle/ds_include_2026-09-05_ge4sections_articles_deepinfra_2048

[OK] Connected to Milvus at http://localhost:19530
[OK] Collection 'DS_INCLUDE_2026_09_02_DEEPINFRA_2048' exists with correct schema (31 fields)
Entity count before : 68,946



Inserting to Milvus: 100%|██████████| 1594/1594 [00:55<00:00, 28.98it/s]


----------------------------------------------------------------------
MILVUS INSERTION SUMMARY
----------------------------------------------------------------------
files_processed     : 1,594
files_skipped       : 0
total_chunks        : 65,677
Entity count after  : 130,326
----------------------------------------------------------------------

insert finished in 0.9 min: {'files_processed': 1594, 'files_skipped': 0, 'total_chunks': 65677}


## Cell 6: verify the new collection

Counts come from `count(*)` (which sees growing segments, unlike `num_entities`), plus
a live vector search using a freshly embedded query, so this validates the whole loop:
query embedding, index, metric, and stored metadata, independent of the API service.


In [17]:
# ---- Cell 6: verification ----
from pymilvus import connections, Collection
from collections import Counter

connections.connect(host="localhost", port=19530)
col = Collection(NEW_COLLECTION)
col.load()

total = col.query(expr="pk >= 0", output_fields=["count(*)"])[0]["count(*)"]
docs, inc, secs = set(), set(), Counter()
it = col.query_iterator(expr="pk >= 0", output_fields=["pmcid", "is_include", "section"],
                        batch_size=16384)
while True:
    rows = it.next()
    if not rows:
        break
    for r in rows:
        docs.add(r["pmcid"])
        secs[r["section"]] += 1
        if r["is_include"] == 1:
            inc.add(r["pmcid"])
it.close()
print(f"collection    : {NEW_COLLECTION}")
print(f"vectors       : {total:,}")
print(f"documents     : {len(docs):,}")
print(f"INCLUDE docs  : {len(inc):,}")
print("chunks/section:", dict(secs.most_common()))

# end-to-end search check with a real query vector
emb = eai.get_embedding_provider()
qv = emb.embed_texts(["What is the prevalence of congenital heart disease in Down syndrome?"])
res = col.search(qv, "vector", {"metric_type": "IP", "params": {"nprobe": 64}},
                 limit=5, output_fields=["pmcid", "section"])
print("\ntop hits for a CHD prevalence query:")
for h in res[0]:
    print(f"  {h.entity.get('pmcid')}:{h.entity.get('section')}  score={h.score:.4f}")
assert len(res[0]) == 5 and res[0][0].score > 0.4, "search sanity check failed"
print("\nsearch OK: the new collection answers queries")


2026-09-06 08:07:33,018 [WARNING][__setup_ts_by_request]: failed to get mvccTs from milvus server, use client-side ts instead (iterator.py:259)


collection    : DS_INCLUDE_2026_09_02_DEEPINFRA_2048
vectors       : 65,677
documents     : 1,594
INCLUDE docs  : 158
chunks/section: {'Results': 19886, 'Methods': 16811, 'Discussion': 16268, 'Introduction': 7932, 'Abstract': 3499, 'Conclusion': 1281}
[OK] DeepInfra embeddings initialized (model: nvidia/llama-nemotron-embed-vl-1b-v2, batch: 32)

top hits for a CHD prevalence query:
  PMC6389118:Conclusion  score=0.7399
  PMC10496550:Results  score=0.7291
  PMC13137511:Discussion  score=0.7069
  PMC10496550:Abstract  score=0.7000
  PMC10496550:Abstract  score=0.6822

search OK: the new collection answers queries


---
## Production state and rollback

The collection this notebook maintains, `DS_INCLUDE_2026_09_02_DEEPINFRA_2048`, is the
live collection named in `srv/.env`; no cutover step is needed after a converging
rerun, only a cache clear (`curl -X POST http://localhost:8698/cache/clear`).

2026-09-06: the collection was purged to articles only
(`scripts/purge_non_articles_2026_09_06.py --apply` deleted the 3,269 chunks of the 75
non-article papers), and Cell 2 gained the matching `type == "article"` selection
criterion, so reruns converge to the same article-only state. The 2026-09-05 pipeline
rerun directory was hardlink-merged into `ds_include_pipeline_2026-09-02/` (identical
content) and archived; 2026-09-02 is the canonical build directory. The rollback
pickle sets (pre-purge 1,669-paper, full 1,765-paper, and the v7 1,501-paper set) are
archived under `output/_archive_superseded_2026-09-06/embeddings_pickle/`; the only
set kept live is this notebook's 1,594-article `PICKLE_DIR`.

Rollback to the archived pre-selection corpus: run
`scripts/rename_collections_2026_09_06.py` with `ROLLBACK = True`, then
`sudo docker compose restart api` and clear the cache.

Cutover for a FUTURE corpus (new snapshot date, hence a new collection name):

1. Edit `srv/.env`: set `MILVUS_COLLECTION` to the new collection name, keeping the old
   value in a comment for rollback.
2. `sudo docker compose up -d api ui` (a plain `restart` is NOT enough for `.env`
   changes: the container's env is baked at creation from `env_file:`; see CLAUDE.md,
   "Apply .env changes").
3. `curl -X POST http://localhost:8698/cache/clear`
4. `curl http://localhost:8698/health` and confirm `total_vectors`,
   `estimated_documents` and `include_count` match Cell 6.
5. Ask the UI a known question and check citations resolve.

Note for the UI FAQ fallbacks: `_FAQ_FALLBACK_DOCS`, `_FAQ_FALLBACK_INCLUDE` and
`_FAQ_FALLBACK_VECTORS` in `ui_streamlit_ds_chat.py` should be updated whenever the
corpus changes (they only matter when /health is unreachable). Keep `PICKLE_DIR` in
`docker-compose.yml` pointed at this notebook's `PICKLE_DIR` (the startup
empty-collection auto-insert recovery path), and re-run
`scripts/export_corpus_manifest.py` so the GUI's downloadable corpus list stays
current.
